In [ ]:
############# Step 0. Merge and generate the ortholog name files for filtering #############

import os
import glob
import pandas as pd

INPUT_DIR = "./Orthologs/orthlogNames"
OUTPUT_DIR = "./Orthologs/orthlogNames"

search_pattern = os.path.join(INPUT_DIR, "*_orthologs_extracted.csv")
csv_files = glob.glob(search_pattern)

if not csv_files:
    print("cannot find csv.")
else:
    print(f"Total {len(csv_files)} are found")
    
    # load the first file (delete the replicates to prevent data explosion)
    merged_df = pd.read_csv(csv_files[0])
    merged_df = merged_df.drop_duplicates(subset=['qseqid']) 
    
    # Merge with rest csvs
    for file in csv_files[1:]:
        df_temp = pd.read_csv(file)
        df_temp = df_temp.drop_duplicates(subset=['qseqid']) 
        
        cols_to_use = df_temp.columns.difference(merged_df.columns).tolist()
        cols_to_use.append('qseqid')
        
        merged_df = pd.merge(merged_df, df_temp[cols_to_use], on='qseqid', how='inner')
    
    # NaN/absent filtering
    genotypes = ['genotype1', 'genotype2', 'genotype3']
    existing_genotypes = [g for g in genotypes if g in merged_df.columns]
    
    for geno in existing_genotypes:
        merged_df = merged_df[
            merged_df[geno].notna() & 
            ~merged_df[geno].astype(str).str.lower().str.contains('absent')
        ]
        
    print(f"Filtering completed. Retained candidates: {len(merged_df)}")
    
    # Save the file
    output_filepath = os.path.join(OUTPUT_DIR, "shared_candidates_intersection.csv")
    merged_df.to_csv(output_filepath, index=False)
    print(f"File saved in: {output_filepath}")